### ЗАДАЧА: Пакетная загрузка накладных (try/except + custom exceptions)

Из внешней системы приходят строки с накладными.
Нужно безопасно разобрать их, отделить валидные записи от ошибок и собрать краткий отчёт.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `InvoiceError`
   - `RowFormatError`
   - `QuantityError`
   - `PriceError`
   - `StatusError`.

2. Функцию `parse_invoice(row)`:
   - формат: `invoice_id,item,quantity,price,status`
   - `quantity` должен быть целым числом и `> 0`
   - `price` должен быть числом и `> 0`
   - допустимые статусы: `new`, `approved`, `paid`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `load_invoices(rows)`:
   - вернуть `(invoices, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных накладных,
   - ошибки по типам,
   - сумму только для накладных со статусом `paid`,
   - товар-лидер по суммарному количеству в валидных накладных.

ПОДСКАЗКИ:
- `quantity` и `price` сначала конвертируются, потом валидируются.
- Для ошибок по типам удобно собирать обычный словарь-счётчик.


In [8]:
rows = [
    'INV-100,Keyboard,3,120,paid',
    'INV-101,Mouse,0,40,new',
    'INV-102,Monitor,2,abc,approved',
    'INV-103,Laptop,1,1400,shipped',
    'INV-104,Keyboard,5,110,paid',
    'INV-105,Dock,2,-50,approved',
]


class InvoiceError(Exception):
    pass


class RowFormatError(InvoiceError):
    pass


class QuantityError(InvoiceError):
    pass


class PriceError(InvoiceError):
    pass


class StatusError(InvoiceError):
    pass


def parse_invoice(row):
    # TODO: распарсить строку и провалидировать quantity, price, status
    # TODO: при ошибках конвертации использовать raise ... from ...
    invoice_id,item,quantity,price,status = row.split(",")
    try:
        quantity = int(quantity)
    except ValueError:
        raise QuantityError("Число должно быть целым")
    
    if quantity <= 0:
        raise QuantityError("Число не должно быть больше 0")
    
    try:
        price = float(price)
    except ValueError:
        raise PriceError("Должен быть числом")
    
    if price <= 0:
        raise PriceError("Число не должно быть отрицательным")
    
    if status not in ["new", "approved", "paid"]:
        raise StatusError(f"Должен быть числом: {status}. Допустимые значения: new, approved, paid")
    
    return {"invoice_id": invoice_id, "item": item, "quantity": quantity, "price": price, "status": status}
    

def load_invoices(rows):
    # TODO: вернуть (invoices, errors)
    invoices = []
    errors = []

    for row in rows:
        try:
            invoices.append(parse_invoice(row))
        except InvoiceError as e:
            errors.append((row, type(e).__name__, e))
    return invoices, errors


# TODO: вызвать load_invoices(rows)
invoices, errors = load_invoices(rows)


# TODO: вывести число валидных накладных и число ошибок
print(f"Число валидных накладных: {len(invoices)}")
print(f"Число ошибок: {len(errors)}")

print(invoices)
# TODO: вывести ошибки по типам
error_types = {}
for _,error_type,_ in errors:
    error_types[error_type] = error_types.get(error_type, 0) + 1
print(error_types)




# TODO: посчитать paid_total
paid_total = sum(inv["price"] * inv["quantity"] for inv in invoices if inv["status"] == "paid")


# TODO: найти товар-лидер по количеству
item_counts = {}
for invoice in invoices:
    item_counts[invoice["quantity"]] = invoice["item"]

print(max(item_counts.values(), key=item_counts.get))
for invoice in invoices:
    item = invoice["item"]
    item_counts["item"] = item_counts.get(item, 0) + invoice["quantity"]
    

if item_counts:
    leader_item = max(item_counts, key=item_counts.get)
    leader_count = item_counts[leader_item]
    print(f"Товар-лидер по количеству: {leader_item} ({leader_count} штук)")
else:
    print("Нет валидных накладных")


Число валидных накладных: 2
Число ошибок: 4
[{'invoice_id': 'INV-100', 'item': 'Keyboard', 'quantity': 3, 'price': 120.0, 'status': 'paid'}, {'invoice_id': 'INV-104', 'item': 'Keyboard', 'quantity': 5, 'price': 110.0, 'status': 'paid'}]
{'QuantityError': 1, 'PriceError': 2, 'StatusError': 1}


TypeError: '>' not supported between instances of 'NoneType' and 'NoneType'